In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Resonance

Have you ever pushed a kid on a swing?  You don't shove whenever you feel like it.  You wait until the swing comes back to you, and *then* you push, a little, every time.  A bunch of small, well-timed pushes builds up a huge swing.  Push at the wrong times and you mostly fight the swing instead.

That's **resonance**: a system that likes to oscillate at some natural frequency, pushed by something that oscillates too.  When the two frequencies match, small pushes add up to big motion.  It's why soldiers break step crossing bridges, how a radio picks one station out of all the others, and why a wine glass can shatter when someone sings the right note.

So far every system we've simulated has been left alone: we set it going and watched.  Today we'll start *pushing*.

We'll see:
 - how to describe an outside push in code, separately from the thing being pushed,
 - what a spring does when it's pushed too slowly, too quickly, and just right,
 - how to *guess* the exact answers, and check our guesses,
 - what friction changes: a lag between push and response, and a sharp peak in how much energy gets absorbed,
 - a sneaky way friction costs Verlet some accuracy,
 - and the very same physics in a circuit from E&M.

## Our Tools So Far

Here's what we need from last time, copied over without changes.

In [ ]:
class FirstOrderODE:
    '''
    A first-order ODE specifies the rate of change of a state y(t)
    in terms of
     - the state itself, y
     - the time, t.

    The generic form for an ODE is

    dy
    --  = f(t, y)
    dt

    '''

    def f(self, t, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f.')

    def exact_solution(self, t, *args):
        raise NotImplementedError('The exact solution is not known (or at least not provided).')

In [ ]:
class SecondOrderODE(FirstOrderODE):
    '''
    A second-order ODE specifies the acceleration

    d^2x
    ----  = a(t, x, v)
    dt^2

    where v = dx/dt is the velocity.  Treating the state as y = (x, v) turns it into
    a first-order ODE,

    d  ( x )   (    v     )
    -- (   ) = (          )
    dt ( v )   ( a(t,x,v) )

    so a subclass only needs to provide the acceleration.
    '''

    def acceleration(self, t, x, v):
        raise NotImplementedError('The generic second-order ODE has an unspecified acceleration.')

    def f(self, t, y):
        # Unpack the state into its two components,
        x, v = y
        # and pack their rates of change back up the same way.
        return np.array([v, self.acceleration(t, x, v)])

In [ ]:
class FirstOrderODEIntegrator:
    '''
    An integrator is an algorithm for producing numerical solutions to ODEs.
    '''

    def __init__(self, ODE):

        self.ODE = ODE

    def starting_from(self, t_0, y_0):
        raise NotImplementedError('No integration algorithm has been specified.')

In [ ]:
class VelocityVerlet(FirstOrderODEIntegrator):
    '''
    Velocity Verlet splits each step into a half kick, a drift, and another half kick,

    v_{n+1/2} = v_n       + 1/2 a(t_n, x_n) Δt
    x_{n+1}   = x_n       + v_{n+1/2} Δt
    v_{n+1}   = v_{n+1/2} + 1/2 a(t_{n+1}, x_{n+1}) Δt

    It needs the ODE's acceleration, so it only works on second-order ODEs.
    '''

    def __init__(self, ODE, Delta_t, steps):

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):

        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array with room for a whole state at each time.
        y = np.zeros((self.steps+1,) + np.shape(y_0))

        # The user picked a starting state
        y[0] = y_0

        for step in range(self.steps):
            x, v = y[step]
            # (acceleration also wants a velocity; we hand it the newest one we have.
            #  For forces that depend only on position it doesn't matter.)
            v_half = v      + 0.5 * self.ODE.acceleration(t[step],   x,     v)      * self.Delta_t   # half kick
            x_new  = x      +       v_half                                           * self.Delta_t   # drift
            v_new  = v_half + 0.5 * self.ODE.acceleration(t[step+1], x_new, v_half) * self.Delta_t   # half kick
            y[step+1] = np.array([x_new, v_new])

        return t, y

# Pushing on a Spring

Recall that the harmonic oscillator has the acceleration

$$
\ddot{x} = -\omega_0^2 x
$$

where $\omega_0$ is its **natural frequency**, the frequency it swings at when left alone.  We'll put a subscript 0 on it from now on, because there's about to be a second frequency in the problem.

Now let's push on it with an outside force $F(t)$.  Newton says the push just adds to the acceleration,

$$
\ddot{x} = -\omega_0^2 x + \frac{F(t)}{m}.
$$

While we're at it, let's allow some friction too: a drag force proportional to the velocity, $-m\gamma v$, with $\gamma$ a rate that says how strong the friction is,

$$
\ddot{x} = -\omega_0^2 x - \gamma \dot{x} + \frac{F(t)}{m}.
$$

Setting $\gamma = 0$ turns friction off, which is where we'll start.

Notice that this system is *not* conservative anymore.  The force depends on the time (and, with friction, on the velocity), so it isn't minus the slope of any potential.  That's the whole point!  The push is supposed to put energy in, and the friction takes energy out.

## The Drive

Here's a design question.  The spring is one thing and the pushing is another.  We could push on our spring in lots of ways: smoothly, in sudden kicks, slowly, quickly...  And the same push could be applied to different springs.

So let's keep them separate.  A `Drive` knows how hard it pushes at each time, and nothing else.  We'll hand a drive to our oscillator, and the oscillator will just ask it for the force whenever it needs one.  If we want to push differently, we hand over a different drive and don't touch the oscillator at all.

The drive can also hold onto the exact solution, if we know one.  The exact motion depends on both the spring and the push, so it needs to be told about the oscillator, too.

In [ ]:
class Drive:
    '''
    A drive is an outside push that depends only on the time.

    Its force is per unit mass, F(t)/m, like everything else.
    '''

    def force(self, t):
        raise NotImplementedError('The generic drive has an unspecified force.')

    def exact_solution(self, oscillator, t, x_0, v_0):
        raise NotImplementedError('The exact motion under this drive is not known (or at least not provided).')

The simplest smooth push is a cosine, with a strength $f$ (force per unit mass) and its own frequency $\Omega$, the **driving frequency**,

$$
\frac{F(t)}{m} = f \cos(\Omega t).
$$

$\Omega$ is a capital $\omega$, so we don't confuse the driving frequency with the natural frequency $\omega_0$ of the spring.

In [ ]:
class Sinusoid(Drive):
    '''
    A sinusoidal drive pushes with force per unit mass

    F(t)/m = f cos(Ω t)

    where Ω is the driving frequency.  Ω may be an array, to drive many copies at once.
    '''

    def __init__(self, f, Omega):
        self.f = f
        self.Omega = Omega

    def force(self, t):
        return self.f * np.cos(self.Omega * t)

## The Driven Oscillator

Now the spring.  It's a `SecondOrderODE`, so all it needs is an acceleration.  It isn't a `ConservativeSystem`, but it's still nice to know how much energy is stored in the spring itself,

$$
E = \frac{1}{2} v^2 + \frac{1}{2}\omega_0^2 x^2
$$

(per unit mass).  Now that's not conserved: watching it change is how we'll see the drive pumping energy in.

In [ ]:
class DrivenOscillator(SecondOrderODE):
    '''
    A harmonic oscillator with friction, pushed by a drive:

    a = -ω_0^2 x - γ v + F(t)/m

    where ω_0 is the natural frequency and γ sets the strength of the friction.
    Everything is per unit mass.
    '''

    def __init__(self, omega, gamma, drive):
        self.omega = omega
        self.gamma = gamma
        self.drive = drive

    def acceleration(self, t, x, v):
        return -self.omega**2 * x - self.gamma * v + self.drive.force(t)

    def energy(self, x, v):
        # The energy stored in the spring and the motion; the drive and the friction change it.
        return 0.5 * v**2 + 0.5 * self.omega**2 * x**2

    def exact_solution(self, t, x_0, v_0):
        # The exact motion depends on how we're pushed, so we ask the drive.
        return self.drive.exact_solution(self, t, x_0, v_0)

# Pushing Without Friction

Let's push a frictionless spring, starting from rest, at a few different driving frequencies: one well below the natural frequency, one well above, one close to it, and one *exactly* on it.

Since `Sinusoid` is happy to take an array of $\Omega$s, we can push all four copies of the spring in one run.

In [ ]:
omega_0 = 1.0
period  = 2 * np.pi / omega_0       # one natural swing
f       = 1.0

# too slow, too fast, close, and exactly on resonance.
Omega = omega_0 * np.array([0.3, 3.0, 0.9, 1.0])

Delta_t = 0.01 / omega_0
swings  = 30
steps   = int(swings * period / Delta_t)

frictionless = DrivenOscillator(omega_0, 0.0, Sinusoid(f, Omega))
start_at_rest = np.zeros((2, len(Omega)))
t, y = VelocityVerlet(frictionless, Delta_t, steps).starting_from(0.0, start_at_rest)
x, v = y[:, 0], y[:, 1]

print(f'Δt = {Delta_t:g}, for {swings} natural swings ({steps} steps)')
print('Ω / ω_0 =', Omega / omega_0)

How should we compare the push with the motion?  A force and a position aren't the same kind of thing.  But here's a fair comparison.  If you pushed *really* slowly on a spring, it'd just stretch until the spring force balanced your push, $k x = F$, or

$$
x = \frac{F}{k} = \frac{F/m}{\omega_0^2}.
$$

Let's call that the **static stretch**: where the push would hold the spring if there were no hurry.  It's a position, so we can draw it right on top of $x(t)$.

In [ ]:
static_stretch = frictionless.drive.force(t[:, np.newaxis]) / omega_0**2

def compare(ax, which, label):
    ax.plot(t, x[:, which], label='$x(t)$')
    ax.plot(t, static_stretch[:, which], linestyle='--', color='gray', label='static stretch')
    ax.set_title(f'{label}: $\\Omega$ = {Omega[which] / omega_0:g} $\\omega_0$')
    ax.set_ylabel('$x$')

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
compare(ax[0], 0, 'too slow')
compare(ax[1], 1, 'too fast')
ax[0].legend()
ax[1].set_xlabel('$t$')
ax[1].set_xlim(0, 4 * period)
plt.show()

When we push too slowly, the spring sort of follows the static stretch, but there's a wiggle on top that's every bit as big.  When we push too fast, the spring barely moves at all, much less than the static stretch.  In both cases it's hard to tell whether the spring is moving with the push or against it, because the motion is a mix of two swings: one at the push's frequency, and one at the spring's own.  We'll untangle them in a moment.

Now the interesting ones.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
compare(ax[0], 2, 'close')
compare(ax[1], 3, 'exactly on resonance')
ax[0].legend()
ax[1].set_xlabel('$t$')
plt.show()

Close to resonance, the swing grows big, then dies back down to nothing, then grows again...  Those are **beats**, just like the coupled pendulums handing their swing back and forth last time.  Exactly on resonance, it just keeps growing.  It looks like the envelope is a straight line!  Every push lands at just the right moment, so every push adds a little more.

Let's understand all of this.

## Guess and Check

We want to solve

$$
\ddot{x} + \omega_0^2 x = f\cos(\Omega t).
$$

A guess: when you push at frequency $\Omega$, maybe the spring just moves at frequency $\Omega$ too.  Let's try $x = A\cos(\Omega t)$ and see if we can pick $A$ to make it work.  Then $\ddot{x} = -A\Omega^2\cos(\Omega t)$, and plugging in,

$$
-A\Omega^2\cos(\Omega t) + \omega_0^2 A\cos(\Omega t) = f\cos(\Omega t).
$$

Every term has a $\cos(\Omega t)$, so it cancels, and

$$
A = \frac{f}{\omega_0^2 - \Omega^2}.
$$

It works!  Look at what it tells us.
 - When $\Omega \ll \omega_0$, $A \approx f/\omega_0^2$, which is the static stretch.  Push slowly, and the spring keeps up.
 - When $\Omega \gg \omega_0$, $A \approx -f/\Omega^2$: small, and **negative**.  The spring moves *opposite* to the push!  Push fast enough and the spring hardly matters; it's just a mass being shaken, $\ddot{x} \approx f\cos(\Omega t)$, and the position of a shaken mass is opposite its acceleration.
 - When $\Omega \to \omega_0$, $A$ blows up.  That's resonance.

The sign flip is a jump in *phase*: below resonance the spring is in step with the push, and above it's exactly half a cycle (an angle $\pi$) out of step.

### But it doesn't start at rest

Our guess starts at $x = A$, not at rest.  How do we fix that?

Here's the key observation.  Suppose we add on a plain, unpushed swing, $B\cos(\omega_0 t)$.  Its acceleration is $-\omega_0^2 B\cos(\omega_0 t)$, which exactly cancels its $\omega_0^2 x$ term.  So it adds *nothing* to the left side, and the equation is still happy!  We can add any free swing we like.  Picking $B = -A$ makes it start at rest:

$$
x(t) = A\left[\cos(\Omega t) - \cos(\omega_0 t)\right].
$$

That's what we saw: the push's own motion, plus a wiggle at the natural frequency.  Without friction, nothing ever damps that wiggle out.

Does it agree with Verlet?

In [ ]:
A = f / (omega_0**2 - Omega[:3]**2)
exact = A * (np.cos(Omega[:3] * t[:, np.newaxis]) - np.cos(omega_0 * t[:, np.newaxis]))

for which in range(3):
    worst = np.max(np.abs(x[:, which] - exact[:, which])) / np.max(np.abs(exact[:, which]))
    print(f'Ω = {Omega[which] / omega_0:g} ω_0:   largest difference from the guess is {worst:.1e} of the biggest swing')

fig, ax = plt.subplots(3, 1, figsize=(10, 8), layout='constrained')
for which, label in enumerate(['too slow', 'too fast', 'close']):
    ax[which].plot(t, x[:, which], label='Verlet')
    ax[which].plot(t, exact[:, which], linestyle=':', color='black', label='guess')
    ax[which].set_title(f'{label}: $\\Omega$ = {Omega[which] / omega_0:g} $\\omega_0$')
    ax[which].set_ylabel('$x$')
ax[0].set_xlim(0, 4 * period)
ax[1].set_xlim(0, 4 * period)
ax[0].legend(loc='upper right')
ax[-1].set_xlabel('$t$')
plt.show()

Very nice.  We can also check the guess by starting the spring *exactly* where the guess starts, at $x_0 = A$ and $v_0 = 0$.  Then there should be no wiggle at all, just a clean $A\cos(\Omega t)$, in step with the push below resonance and opposite it above.

In [ ]:
slow_and_fast = DrivenOscillator(omega_0, 0.0, Sinusoid(f, Omega[:2]))
start_on_guess = np.array([A[:2], np.zeros(2)])
t_guess, y_guess = VelocityVerlet(slow_and_fast, Delta_t, steps).starting_from(0.0, start_on_guess)

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for which, label in enumerate(['too slow', 'too fast']):
    ax[which].plot(t_guess, y_guess[:, 0, which], label='$x(t)$, starting at $A$')
    ax[which].plot(t_guess, f * np.cos(Omega[which] * t_guess) / omega_0**2, linestyle='--', color='gray', label='static stretch')
    ax[which].set_title(f'{label}: $\\Omega$ = {Omega[which] / omega_0:g} $\\omega_0$')
    ax[which].set_ylabel('$x$')
ax[0].legend()
ax[1].set_xlabel('$t$')
ax[1].set_xlim(0, 4 * period)
plt.show()

### Beats

Near resonance, $\cos(\Omega t) - \cos(\omega_0 t)$ is the difference of two cosines with nearly the same frequency.  There's a trig identity for that,

$$
\cos a - \cos b = -2 \sin\left(\frac{a + b}{2}\right)\sin\left(\frac{a - b}{2}\right),
$$

so

$$
x(t) = 2A \sin\left(\frac{\omega_0 - \Omega}{2}\,t\right)\sin\left(\frac{\omega_0 + \Omega}{2}\,t\right).
$$

The second sine swings at the *average* of the two frequencies, which is quick.  The first sine swings at *half the difference*, which is slow.  The slow one is the **envelope**: it squeezes the quick swing up and down.  The swing dies to nothing every time the envelope crosses zero, which happens every $2\pi / |\omega_0 - \Omega|$.  The closer we push to resonance, the slower the beats, and the bigger they get.

### Exactly on resonance

Exactly on resonance $A$ is infinite and $\cos(\Omega t) - \cos(\omega_0 t)$ is zero, so the formula is $\infty \times 0$.  Not helpful!  Let's sneak up on it instead.  Push at $\Omega = \omega_0 + \epsilon$ with $\epsilon$ tiny.  Taylor expanding in $\epsilon$,

$$
\cos(\Omega t) \approx \cos(\omega_0 t) - \epsilon\, t \sin(\omega_0 t)
\qquad\qquad
\omega_0^2 - \Omega^2 \approx -2\omega_0\epsilon,
$$

so

$$
x(t) \approx \frac{f}{-2\omega_0\epsilon}\left(-\epsilon\, t\sin(\omega_0 t)\right) = \frac{f\,t}{2\omega_0}\sin(\omega_0 t).
$$

The $\epsilon$s cancel, and the $0 \times \infty$ turns into a $t$: the envelope really is a straight line, growing forever.  Let's draw both envelopes on the simulations.

In [ ]:
envelope_close = 2 * A[2] * np.sin((omega_0 - Omega[2]) / 2 * t)
envelope_resonant = f * t / (2 * omega_0)

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(t, x[:, 2], label='$x(t)$')
ax[0].plot(t, +envelope_close, color='black', label='envelope')
ax[0].plot(t, -envelope_close, color='black')
ax[0].set_title(f'close: $\\Omega$ = {Omega[2] / omega_0:g} $\\omega_0$')
ax[1].plot(t, x[:, 3])
ax[1].plot(t, +envelope_resonant, color='black')
ax[1].plot(t, -envelope_resonant, color='black')
ax[1].set_title('exactly on resonance')
for a in ax:
    a.set_ylabel('$x$')
ax[0].legend()
ax[1].set_xlabel('$t$')
plt.show()

worst = np.max(np.abs(x[:, 3] - envelope_resonant * np.sin(omega_0 * t))) / np.max(envelope_resonant)
print(f'On resonance, the largest difference from (f t / 2ω_0) sin(ω_0 t) is {worst:.1e} of the biggest swing')

### Superposition

That trick of adding a free swing is worth checking directly, because it's really useful.  It says that if we push two copies of the spring the same way but start them differently, then the *difference* between them is just a free, unpushed swing.  The push cancels out of the difference!

Let's push two copies at the same $\Omega$, one starting from rest and one not.

In [ ]:
pushed_twice = DrivenOscillator(omega_0, 0.0, Sinusoid(f, Omega[2]))
x_0, v_0 = 0.5, 0.3 * omega_0
two_starts = np.array([[0.0, x_0], [0.0, v_0]])     # columns are the two copies
t_two, y_two = VelocityVerlet(pushed_twice, Delta_t, steps).starting_from(0.0, two_starts)

difference = y_two[:, 0, 1] - y_two[:, 0, 0]
free_swing = x_0 * np.cos(omega_0 * t_two) + v_0 / omega_0 * np.sin(omega_0 * t_two)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_two, difference, label='difference of the two pushed copies')
ax.plot(t_two, free_swing, linestyle='--', color='black', label='free swing')
ax.set_xlabel('$t$')
ax.set_ylabel('$x$')
ax.set_xlim(0, 4 * period)
ax.legend()
plt.show()

print(f'Largest difference from the free swing: {np.max(np.abs(difference - free_swing)):.1e}')

The difference agrees with the free swing to about Verlet's own error after this many steps, the same size as the other comparisons above.

That works because every term in our equation is just *proportional* to $x$ (or its derivatives).  Equations like that are called **linear**, and "you can add solutions" is called **superposition**.  It wouldn't work for a pendulum, with its $\sin\theta$!

## Where Does the Energy Go?

The push does work on the spring, so the spring's energy changes.  How much energy does pushing put in?

In [ ]:
E = frictionless.energy(x, v)

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for which, label in enumerate(['too slow', 'too fast', 'close']):
    ax[0].plot(t, E[:, which], label=f'{label}: $\\Omega$ = {Omega[which] / omega_0:g} $\\omega_0$')
ax[0].set_ylabel('$E$')
ax[0].legend()
ax[1].plot(t, E[:, 3], label='exactly on resonance')
ax[1].plot(t, f**2 * t**2 / 8, linestyle='--', color='black', label='$f^2 t^2 / 8$')
ax[1].set_ylabel('$E$')
ax[1].set_xlabel('$t$')
ax[1].legend()
plt.show()

Off resonance, the energy sloshes in and out.  For a while the push happens to be in step with the swing and adds energy, then it drifts out of step and takes it back.  Over a long time, it doesn't add up to anything: there's no steady flow of energy in.

On resonance, the energy grows and grows.  The swing's size grows like $t$, so its energy, which goes like the size squared, grows like $t^2$.  (The swing is $\frac{f t}{2\omega_0}\sin(\omega_0 t)$, and a swing of size $a$ has energy $\frac{1}{2}\omega_0^2 a^2$.  That's where $f^2t^2/8$ comes from.)

So: how much energy, on average, does a push at frequency $\Omega$ put in?  Without friction, that question doesn't have a good answer.  Off resonance it's zero, on resonance it's infinite, and in between it depends on how long you wait.  Real springs aren't like that.  A real swing doesn't grow forever, and doesn't keep its start-up wiggle forever, either.  Something takes energy out: friction.

# Adding Friction

## Guessing the Steady Motion

Let's guess again.  With friction, we shouldn't expect the spring to move exactly in step with the push (or exactly opposite it).  So let's allow a *lag*: guess

$$
x = A \cos(\Omega t - \phi)
$$

where $\phi$ is the **phase lag**, how far behind the push the spring is.  Then

$$
\dot{x} = -A\Omega \sin(\Omega t - \phi)
\qquad\qquad
\ddot{x} = -A\Omega^2 \cos(\Omega t - \phi)
$$

and plugging into $\ddot{x} + \gamma\dot{x} + \omega_0^2 x = f\cos(\Omega t)$ gives

$$
A(\omega_0^2 - \Omega^2)\cos(\Omega t - \phi) - \gamma A \Omega \sin(\Omega t - \phi) = f \cos(\Omega t).
$$

The left side is written in terms of $\Omega t - \phi$ and the right isn't.  Let's fix that with the angle-addition formula, $\Omega t = (\Omega t - \phi) + \phi$:

$$
f\cos(\Omega t) = f\cos\phi \cos(\Omega t - \phi) - f\sin\phi\sin(\Omega t - \phi).
$$

Now both sides are some amount of $\cos(\Omega t - \phi)$ plus some amount of $\sin(\Omega t - \phi)$.  For them to agree at *every* time, the amounts have to match separately:

$$
A(\omega_0^2 - \Omega^2) = f\cos\phi
\qquad\qquad
\gamma A \Omega = f\sin\phi.
$$

Two equations, two unknowns!  Dividing one by the other gets rid of $A$,

$$
\tan\phi = \frac{\gamma\Omega}{\omega_0^2 - \Omega^2},
$$

and squaring both and adding gets rid of $\phi$ (since $\cos^2 + \sin^2 = 1$),

$$
A = \frac{f}{\sqrt{(\omega_0^2 - \Omega^2)^2 + \gamma^2\Omega^2}}.
$$

Check: with $\gamma = 0$ these go back to what we found without friction.  And now, even right on resonance, $A = f/\gamma\omega_0$ is finite.  Friction tames the blow-up.

One careful point about $\phi$.  Both $\gamma A\Omega$ and $f\sin\phi$ are positive, so $\sin\phi > 0$ and $\phi$ is between 0 and $\pi$.  But `np.arctan` only ever returns angles between $-\pi/2$ and $\pi/2$, so above resonance (where $\omega_0^2 - \Omega^2 < 0$) it'd give the wrong answer.  `np.arctan2(y, x)` takes the two pieces separately and knows which quadrant the angle is in.

In [ ]:
def steady_amplitude(omega_0, gamma, f, Omega):
    return f / np.sqrt((omega_0**2 - Omega**2)**2 + (gamma * Omega)**2)

def steady_phase(omega_0, gamma, Omega):
    # arctan2(y, x) is the angle of the point (x, y), so it lands in the right quadrant.
    return np.arctan2(gamma * Omega, omega_0**2 - Omega**2)

## Guessing the Start-Up Wiggles

Just like before, the steady guess doesn't start where we want, and we fix that by adding a solution of the *unpushed* problem, $\ddot{x} + \gamma\dot{x} + \omega_0^2 x = 0$.  But now the unpushed spring has friction, so it rings down.  What does that look like?

Here's a trick.  Let's guess the ringing is some decaying exponential times something else, $x = e^{-kt}u(t)$, and see what $u$ has to do.  By the product rule,

$$
\dot{x} = e^{-kt}\left(\dot{u} - k u\right)
\qquad\qquad
\ddot{x} = e^{-kt}\left(\ddot{u} - 2k\dot{u} + k^2 u\right)
$$

and plugging in (every term has an $e^{-kt}$, so it cancels),

$$
\ddot{u} + (\gamma - 2k)\dot{u} + (\omega_0^2 - \gamma k + k^2) u = 0.
$$

We get to pick $k$.  Picking $k = \gamma/2$ kills the $\dot{u}$ term, leaving

$$
\ddot{u} = -\left(\omega_0^2 - \frac{\gamma^2}{4}\right) u,
$$

which is just a plain harmonic oscillator!  We know how to solve that: $u$ swings with frequency

$$
\omega_d = \sqrt{\omega_0^2 - \frac{\gamma^2}{4}},
$$

so the ringing is

$$
x = e^{-\gamma t/2}\left(B\cos(\omega_d t) + C\sin(\omega_d t)\right),
$$

a wiggle that dies away on a timescale of $2/\gamma$.  (This only works if $\gamma < 2\omega_0$, so $\omega_d$ is real.  With more friction than that the spring doesn't wiggle at all, it just oozes back.  We'll stay below.)

Picking $B$ and $C$ so that the total starts at $x_0$ and $v_0$ is a little algebra: at $t = 0$ the steady part is at $A\cos\phi$ with velocity $A\Omega\sin\phi$, so

$$
B = x_0 - A\cos\phi
\qquad\qquad
C = \frac{v_0 - A\Omega\sin\phi + \frac{\gamma}{2}B}{\omega_d}.
$$

## Teaching the Drive the Answer

Now we know the exact motion of a sinusoidally-pushed spring, with or without friction, from any start.  Let's teach it to `Sinusoid`.  This replaces our earlier `Sinusoid`; the push is the same, but now it knows the answer, too.

Careful, though: redefining a class doesn't change the objects we already made from it.  Springs we built earlier, like `frictionless`, are still holding an *old* `Sinusoid`, which doesn't know the answer.  To use the new `exact_solution`, we have to build new springs.

There's one special case: exactly on resonance with no friction, the steady guess has $A = \infty$, and the answer is the growing $\frac{f t}{2\omega_0}\sin(\omega_0 t)$ we found by Taylor expanding.  When we push many copies at once, only some might be exactly on resonance, so we compute both answers and use `np.where` to pick the right one for each copy.

In [ ]:
class Sinusoid(Drive):
    '''
    A sinusoidal drive pushes with force per unit mass

    F(t)/m = f cos(Ω t)

    where Ω is the driving frequency.  Ω may be an array, to drive many copies at once.
    '''

    def __init__(self, f, Omega):
        self.f = f
        self.Omega = Omega

    def force(self, t):
        return self.f * np.cos(self.Omega * t)

    def exact_solution(self, oscillator, t, x_0, v_0):
        omega_0, gamma = oscillator.omega, oscillator.gamma
        f, Omega = self.f, self.Omega

        if gamma >= 2 * omega_0:
            raise ValueError('We only worked out the exact solution for γ < 2ω_0.')

        # Exactly on resonance with no friction the steady amplitude is infinite.
        # There we swap in a harmless Ω so nothing divides by zero; we throw those copies away below.
        on_resonance = (gamma == 0) & (Omega == omega_0)
        Omega = np.where(on_resonance, 0.5 * omega_0, Omega)

        # The steady motion, A cos(Ωt - φ),
        A   = steady_amplitude(omega_0, gamma, f, Omega)
        phi = steady_phase(omega_0, gamma, Omega)
        x_steady =  A *         np.cos(Omega * t - phi)
        v_steady = -A * Omega * np.sin(Omega * t - phi)

        # plus the ringing that makes it start at x_0 and v_0.
        omega_d = np.sqrt(omega_0**2 - gamma**2 / 4)
        B = x_0 - A * np.cos(phi)
        C = (v_0 - A * Omega * np.sin(phi) + gamma / 2 * B) / omega_d
        decay = np.exp(-gamma * t / 2)
        x_ring = decay * (B * np.cos(omega_d * t) + C * np.sin(omega_d * t))
        v_ring = decay * ((omega_d * C - gamma / 2 * B) * np.cos(omega_d * t)
                        - (omega_d * B + gamma / 2 * C) * np.sin(omega_d * t))

        # Exactly on resonance with no friction, the free swing plus the growing response.
        x_resonant = (x_0 * np.cos(omega_0 * t) + v_0 / omega_0 * np.sin(omega_0 * t)
                      + f * t / (2 * omega_0) * np.sin(omega_0 * t))
        v_resonant = (-x_0 * omega_0 * np.sin(omega_0 * t) + v_0 * np.cos(omega_0 * t)
                      + f / (2 * omega_0) * (np.sin(omega_0 * t) + omega_0 * t * np.cos(omega_0 * t)))

        x = np.where(on_resonance, x_resonant, x_steady + x_ring)
        v = np.where(on_resonance, v_resonant, v_steady + v_ring)
        return np.array([x, v])

## Pushing With Friction

Let's push our four springs again, the same four ways, but with some friction this time.  We'll draw the exact answer on top, too.

In [ ]:
gamma = 0.1 * omega_0

rubbing = DrivenOscillator(omega_0, gamma, Sinusoid(f, Omega))
t, y = VelocityVerlet(rubbing, Delta_t, steps).starting_from(0.0, start_at_rest)
x, v = y[:, 0], y[:, 1]
x_exact, v_exact = rubbing.exact_solution(t[:, np.newaxis], 0.0, 0.0)

print(f'γ = {gamma / omega_0:g} ω_0, so the start-up wiggles die away over about 2/γ = {2 / gamma:g}, or {2 / gamma / period:.1f} natural swings')

fig, ax = plt.subplots(4, 1, figsize=(10, 11), layout='constrained')
for which, label in enumerate(['too slow', 'too fast', 'close', 'exactly on resonance']):
    ax[which].plot(t, x[:, which], label='Verlet')
    ax[which].plot(t, x_exact[:, which], linestyle=':', color='black', label='exact')
    ax[which].plot(t, f * np.cos(Omega[which] * t) / omega_0**2, linestyle='--', color='gray', label='static stretch')
    ax[which].set_title(f'{label}: $\\Omega$ = {Omega[which] / omega_0:g} $\\omega_0$')
    ax[which].set_ylabel('$x$')
ax[0].legend(loc='upper right')
# The fast push is a blur over the whole run, so let's zoom in on its last few swings.
ax[1].set_xlim(t[-1] - 4 * period, t[-1])
ax[1].set_title(ax[1].get_title() + ', last few swings')
ax[-1].set_xlabel('$t$')
plt.show()

print(f'Largest difference between Verlet and exact: {np.max(np.abs(x - x_exact)) / np.max(np.abs(x_exact)):.1e} of the biggest swing')

Now the start-up wiggles die away, and after a while each spring settles into a **steady state**, swinging at the driver's frequency with a fixed size.  Even exactly on resonance, the swing stops growing: the push puts energy in, and once the swing is big enough the friction takes it out just as fast.

Too slow, the spring still rides along with the static stretch.  Too fast, it still moves a little, opposite the push.  And on resonance?  The swing is so much bigger than the static stretch that it's hard to compare.  Let's zoom in on the last few swings and shrink each curve to the same height.

In [ ]:
on_resonance = 3
last_swings = t > t[-1] - 3 * period

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t[last_swings], x[last_swings, on_resonance] / np.max(x[last_swings, on_resonance]), label='$x(t)$, shrunk')
ax.plot(t[last_swings], np.cos(Omega[on_resonance] * t[last_swings]), linestyle='--', color='gray', label='push, shrunk')
ax.set_title('exactly on resonance, last few swings')
ax.set_xlabel('$t$')
ax.legend(loc='upper right')
plt.show()

The swing peaks a quarter of a cycle *after* the push does.  It's neither in step nor opposite, but halfway in between.  We'll see why that matters in a moment.

## How Big?

Let's push lots of springs, with a whole range of $\Omega$s, wait for the wiggles to die, and measure how big the steady swing is.

How long do we have to wait?  The wiggles shrink like $e^{-\gamma t/2}$, so if we wait several times $2/\gamma$ they'll be long gone.  (The code prints how much they've shrunk.)

Later we'll try a few different amounts of friction, too.  More friction makes a wider peak, so let's scan a wide enough range of $\Omega$ to see the whole peak for the most friction we'll use.

In [ ]:
gammas = omega_0 * np.array([0.05, 0.1, 0.2])      # the amounts of friction we'll compare later

# Look at least halfway to ω_0 on either side, and wider if the friction makes the peak wide:
# the peak's half-width is γ/2, so we look out to at least five half-widths.
width = max(0.5 * omega_0, 2.5 * np.max(gammas))
Omega_scan = np.linspace(max(omega_0 - width, 0.05 * omega_0), omega_0 + width, 201)

def push_and_wait(gamma, Omega, cycles):
    '''
    Push springs at every Ω in Omega, starting from rest.  Run until the start-up wiggles
    are long gone, and then for the given number of driving cycles at the slowest Ω.
    Returns t, x, v, and when the wiggles are gone.
    '''
    settled = 10 * 2 / gamma
    t_end = settled + cycles * 2 * np.pi / np.min(Omega)
    spring = DrivenOscillator(omega_0, gamma, Sinusoid(f, Omega))
    t, y = VelocityVerlet(spring, Delta_t, int(t_end / Delta_t)).starting_from(0.0, np.zeros((2, len(Omega))))
    return t, y[:, 0], y[:, 1], settled

cycles = 20
t, x, v, settled = push_and_wait(gamma, Omega_scan, cycles)
steady = t >= settled
print(f'Waited until t = {settled:g}, so the start-up wiggles have shrunk by a factor of {np.exp(-gamma * settled / 2):.1e}')
measured_amplitude = np.max(np.abs(x[steady]), axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(Omega_scan / omega_0, measured_amplitude, marker='.', linestyle='none', label='Verlet')
ax.plot(Omega_scan / omega_0, steady_amplitude(omega_0, gamma, f, Omega_scan), color='black', label='$A(\\Omega)$')
ax.set_xlabel('$\\Omega / \\omega_0$')
ax.set_ylabel('steady swing size')
ax.legend()
plt.show()

A big, narrow peak right at the natural frequency.  That's the picture people usually have in mind when they say "resonance."

## The Lag

Now let's measure the lag $\phi$.  How can we tell, from $x(t)$, how far behind the push the spring is?

Here's an idea.  In the steady state, $x = A\cos(\Omega t - \phi)$.  By the angle-addition formula, that's

$$
x = A\cos\phi\,\cos(\Omega t) + A\sin\phi\,\sin(\Omega t).
$$

So the motion is some amount of "looks like the push", $\cos(\Omega t)$, plus some amount of "a quarter cycle behind the push", $\sin(\Omega t)$.  If we could measure those two amounts, the lag is the angle that splits them.

To measure how much of $x$ looks like $\cos(\Omega t)$, let's multiply by $\cos(\Omega t)$ and average over many whole cycles.  Over a whole cycle, $\cos^2$ averages to $\frac{1}{2}$ and $\sin\cos$ averages to 0, so

$$
\langle x\cos(\Omega t)\rangle = \frac{1}{2}A\cos\phi
\qquad\qquad
\langle x\sin(\Omega t)\rangle = \frac{1}{2}A\sin\phi.
$$

The averages pick out the two amounts, one at a time!  The lag is then `np.arctan2` of the pair, for the same reason as before.  We have to be careful to average over a *whole number* of cycles of each $\Omega$, or the $\sin\cos$ won't average to zero.

In [ ]:
def average_over_cycles(t, quantity, Omega, start, cycles):
    '''
    Average quantity (with time going down the first axis) over the given number of whole
    driving cycles of each Ω, beginning at time start.
    '''
    end = start + cycles * 2 * np.pi / Omega
    window = (t[:, np.newaxis] >= start) & (t[:, np.newaxis] < end)
    return np.sum(window * quantity, axis=0) / np.sum(window, axis=0)

In [ ]:
runs = {g: push_and_wait(g, Omega_scan, cycles) for g in gammas}

fig, ax = plt.subplots(figsize=(10, 4))
for g in gammas:
    t, x, v, settled = runs[g]
    push = np.cos(Omega_scan * t[:, np.newaxis])
    quarter_cycle_behind = np.sin(Omega_scan * t[:, np.newaxis])

    in_step = average_over_cycles(t, x * push, Omega_scan, settled, cycles)
    behind  = average_over_cycles(t, x * quarter_cycle_behind, Omega_scan, settled, cycles)
    measured_phase = np.arctan2(behind, in_step)
    predicted_phase = steady_phase(omega_0, g, Omega_scan)

    line, = ax.plot(Omega_scan / omega_0, predicted_phase, label=f'$\\gamma$ = {g / omega_0:g} $\\omega_0$')
    ax.plot(Omega_scan / omega_0, measured_phase, marker='.', linestyle='none', color=line.get_color())
    print(f'γ = {g / omega_0:g} ω_0:   largest difference between measured and predicted lag is '
          f'{np.max(np.abs(measured_phase - predicted_phase)):.1e} radians')

ax.axhline(np.pi / 2, color='gray', linestyle='--')
ax.axvline(1, color='gray', linestyle='--')
ax.set_yticks([0, np.pi / 2, np.pi], ['0', '$\\pi/2$', '$\\pi$'])
ax.set_xlabel('$\\Omega / \\omega_0$')
ax.set_ylabel('lag $\\phi$')
ax.legend()
plt.show()

Below resonance the spring is in step with the push, $\phi \approx 0$.  Above, it's opposite, $\phi \approx \pi$.  And instead of jumping, the lag sweeps smoothly between them, passing through exactly $\pi/2$, a quarter cycle behind, right at $\Omega = \omega_0$.  (Check the formula: $\tan\phi = \infty$ there.)  The more friction, the more gradual the sweep; the less friction, the closer it gets to the sudden jump we had without any.

## Where Does the Energy Go Now?

The push does work on the spring at a rate (power, per unit mass) of force times velocity,

$$
P = f\cos(\Omega t)\,v.
$$

In the steady state $v = -A\Omega\sin(\Omega t - \phi) = -A\Omega\left[\cos\phi\sin(\Omega t) - \sin\phi\cos(\Omega t)\right]$.  Averaging over whole cycles, the $\sin\cos$ part averages away and the $\cos^2$ part gives a half,

$$
\langle P\rangle = \frac{1}{2} f\,\Omega A\sin\phi.
$$

Only the part of the motion that's a quarter cycle *behind* the push absorbs energy on average!  The in-step part takes energy in for half a cycle and gives it back the other half.  That's why there was no steady flow of energy without friction: with $\phi = 0$ or $\pi$, $\sin\phi = 0$.

We can tidy this up.  From our matching, $\gamma A\Omega = f\sin\phi$, so

$$
\langle P\rangle = \frac{1}{2}\gamma\,\Omega^2 A^2 = \frac{\frac{1}{2}\gamma\,\Omega^2 f^2}{(\omega_0^2 - \Omega^2)^2 + \gamma^2\Omega^2}.
$$

Let's measure the average power for the same three amounts of friction.

In [ ]:
def average_power(omega_0, gamma, f, Omega):
    return 0.5 * gamma * Omega**2 * steady_amplitude(omega_0, gamma, f, Omega)**2

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for g in gammas:
    t, x, v, settled = runs[g]
    power = f * np.cos(Omega_scan * t[:, np.newaxis]) * v
    measured_power = average_over_cycles(t, power, Omega_scan, settled, cycles)
    predicted_power = average_power(omega_0, g, f, Omega_scan)

    line, = ax.plot(Omega_scan / omega_0, predicted_power, label=f'$\\gamma$ = {g / omega_0:g} $\\omega_0$')
    ax.plot(Omega_scan / omega_0, measured_power, marker='.', linestyle='none', color=line.get_color())

    print(f'γ = {g / omega_0:g} ω_0:   largest difference between measured and predicted power is '
          f'{np.max(np.abs(measured_power - predicted_power)) / np.max(predicted_power):.1e} of the peak')

    # Energy balance: in the steady state, the friction must take out exactly what the push puts in.
    friction_loss = average_over_cycles(t, g * v**2, Omega_scan, settled, cycles)
    print(f'                   largest mismatch between power in and friction loss is '
          f'{np.max(np.abs(measured_power - friction_loss)) / np.max(predicted_power):.1e} of the peak')

ax.set_xlabel('$\\Omega / \\omega_0$')
ax.set_ylabel('average power absorbed $\\langle P \\rangle$')
ax.legend()
plt.show()

Less friction makes a taller, narrower peak.  How narrow?  Near resonance, $\Omega \approx \omega_0$, we can approximate $\omega_0^2 - \Omega^2 = (\omega_0 + \Omega)(\omega_0 - \Omega) \approx 2\omega_0(\omega_0 - \Omega)$, and $\Omega \approx \omega_0$ everywhere else.  Then

$$
\langle P\rangle \approx \frac{f^2}{8}\,\frac{\gamma}{(\Omega - \omega_0)^2 + (\gamma/2)^2}.
$$

This shape is called a **Lorentzian**.  It falls to half its peak when $\Omega - \omega_0 = \pm\gamma/2$, so its full width at half its height is exactly $\gamma$.  The friction sets the width!

People often describe how sharp a resonance is with the **quality factor**, $Q = \omega_0/\gamma$: the natural frequency divided by the width.  A high-$Q$ system responds to a very narrow range of driving frequencies, and when left alone it rings for a long time: its swing shrinks by a factor of $e$ only every $Q/\pi$ swings.  A tuning fork or a quartz watch crystal has a huge $Q$ (thousands, or more!); a car's shock absorbers are designed to have a tiny one.

In [ ]:
for g in gammas:
    print(f'γ = {g / omega_0:g} ω_0:   Q = {omega_0 / g:.3g}')

# Verlet Gets Demoted

You might have noticed that Verlet agreed with the exact answers a little less well once we turned on friction.  Is that just because the problem got harder, or is something else going on?

Let's find out how Verlet's error depends on $\Delta t$, with and without friction.  We'll push a spring from rest, run to the same final time with smaller and smaller steps, and compare where it ends up with the exact answer.  Remember from our log-log plots: if the error goes like $\Delta t^p$, it'll be a straight line with slope $p$.

In [ ]:
Omega_test = 0.9 * omega_0
final_time = 50 / omega_0
Delta_ts = 0.2 / omega_0 / 2**np.arange(6)

fig, ax = plt.subplots(figsize=(7, 5))
for g in [0.0, 0.1 * omega_0]:
    spring = DrivenOscillator(omega_0, g, Sinusoid(f, Omega_test))
    x_final_exact, _ = spring.exact_solution(final_time, 0.0, 0.0)

    errors = []
    for dt in Delta_ts:
        t, y = VelocityVerlet(spring, dt, int(round(final_time / dt))).starting_from(0.0, np.array([0.0, 0.0]))
        errors.append(np.abs(y[-1, 0] - x_final_exact))
    errors = np.array(errors)

    slope, _ = np.polyfit(np.log(Delta_ts), np.log(errors), 1)
    ax.loglog(Delta_ts, errors, marker='o', label=f'$\\gamma$ = {g / omega_0:g} $\\omega_0$, slope {slope:.2f}')
    print(f'γ = {g / omega_0:g} ω_0:   error at t = {final_time:g} goes like Δt^{slope:.2f}')

ax.set_xlabel('$\\Delta t$')
ax.set_ylabel('error in $x$ at the final time')
ax.legend()
plt.show()

Without friction, the slope is about 2: Verlet is second order, just like we found before.  But with friction, the slope is about 1!  Verlet got *demoted* to first order.  Halving $\Delta t$ only halves the error, instead of cutting it by four.

What happened?  Look at the last line of a Verlet step:

```python
v_new  = v_half + 0.5 * self.ODE.acceleration(t[step+1], x_new, v_half) * self.Delta_t   # half kick
```

The last half kick should use the acceleration at the *end* of the step, and with friction that depends on the velocity at the end of the step, `v_new`.  But `v_new` is exactly what we're trying to compute!  So Verlet hands over the newest velocity it has, `v_half`, which is off by about half a step's worth of acceleration.  For forces that only depend on position (every force we'd met before today), the velocity doesn't matter and nothing goes wrong.  But friction does depend on the velocity, so every step makes a small mistake, and those mistakes add up to an error proportional to $\Delta t$ instead of $\Delta t^2$.

Remember, too, that Verlet's real superpower was keeping the energy honest for *conservative* systems.  With friction, energy isn't supposed to be conserved anyway, so that superpower doesn't buy us much here.

Is this a disaster?  Not for us.  All the comparisons above used a small enough $\Delta t$ that Verlet agreed with the exact answers to a fraction of a percent.  But it's a good reminder: an algorithm's order isn't a fixed fact about the algorithm.  It depends on the problem you feed it, too.

# The Same Thing in E&M: the RLC Circuit

Here's a circuit you might meet in E&M: a resistor $R$, an inductor $L$, and a capacitor $C$, all in one loop, driven by an AC voltage source $V_0\cos(\Omega t)$.

In [ ]:
def draw_rlc_circuit(ax):
    wire = dict(color='black', linewidth=2)
    width, height = 6.0, 3.0

    # Resistor: a zigzag on the top-left of the loop.
    zig_x = np.linspace(1.0, 2.6, 13)
    zig_y = height + 0.25 * np.array([0, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1, 0])
    ax.plot([0, 1.0], [height, height], **wire)
    ax.plot(zig_x, zig_y, **wire)
    ax.text(1.8, height + 0.45, '$R$', ha='center', fontsize=16)

    # Inductor: a row of loops on the top-right.
    ax.plot([2.6, 3.4], [height, height], **wire)
    s = np.linspace(0, 1, 400)
    turns = 4
    coil_x = 3.4 + 1.6 * s - 0.15 * np.sin(2 * np.pi * turns * s)
    coil_y = height + 0.3 * (1 - np.cos(2 * np.pi * turns * s)) / 2
    ax.plot(coil_x, coil_y, **wire)
    ax.plot([5.0, width], [height, height], **wire)
    ax.text(4.2, height + 0.55, '$L$', ha='center', fontsize=16)

    # Capacitor: two plates on the right side.
    gap = 0.15
    ax.plot([width, width], [height, height / 2 + gap], **wire)
    ax.plot([width, width], [height / 2 - gap, 0], **wire)
    ax.plot([width - 0.5, width + 0.5], [height / 2 + gap] * 2, **wire)
    ax.plot([width - 0.5, width + 0.5], [height / 2 - gap] * 2, **wire)
    ax.text(width + 0.7, height / 2, '$C$', va='center', fontsize=16)

    # Bottom wire.
    ax.plot([width, 0], [0, 0], **wire)

    # AC source: a circle with a wiggle, on the left side.
    radius = 0.45
    ax.plot([0, 0], [0, height / 2 - radius], **wire)
    ax.plot([0, 0], [height / 2 + radius, height], **wire)
    angle = np.linspace(0, 2 * np.pi, 200)
    ax.plot(radius * np.cos(angle), height / 2 + radius * np.sin(angle), **wire)
    wiggle = np.linspace(-0.6, 0.6, 100) * radius
    ax.plot(wiggle, height / 2 + 0.2 * np.sin(np.pi * wiggle / (0.6 * radius)), **wire)
    ax.text(-0.7, height / 2, '$V_0\\cos(\\Omega t)$', ha='right', va='center', fontsize=16)

    # The current, going around clockwise.
    ax.annotate('', xy=(width / 2 + 0.4, -0.35), xytext=(width / 2 - 0.4, -0.35),
                arrowprops=dict(arrowstyle='<-', linewidth=1.5))
    ax.text(width / 2, -0.95, '$I = \\dot{q}$', ha='center', fontsize=16)

    ax.set_xlim(-3.2, width + 1.4)
    ax.set_ylim(-1.4, height + 1.0)
    ax.set_aspect('equal')
    ax.axis('off')

fig, ax = plt.subplots(figsize=(8, 4))
draw_rlc_circuit(ax)
plt.show()

Let $q$ be the charge on the capacitor, so the current around the loop is $I = \dot{q}$.  Kirchhoff's loop rule says the source's voltage gets used up by the three pieces:
 - the resistor takes $RI = R\dot{q}$,
 - the inductor takes $L\dot{I} = L\ddot{q}$,
 - the capacitor takes $q/C$.

So

$$
L\ddot{q} + R\dot{q} + \frac{q}{C} = V_0\cos(\Omega t).
$$

Divide through by $L$:

$$
\ddot{q} + \frac{R}{L}\dot{q} + \frac{1}{LC}q = \frac{V_0}{L}\cos(\Omega t).
$$

That's *exactly* our pushed spring with friction, $\ddot{x} + \gamma\dot{x} + \omega_0^2 x = f\cos(\Omega t)$, with different letters!

| spring | circuit |
| :-- | :-- |
| position $x$ | charge $q$ |
| velocity $v$ | current $I$ |
| natural frequency $\omega_0^2$ | $\dfrac{1}{LC}$ |
| friction $\gamma$ | $\dfrac{R}{L}$ |
| push $f$ | $\dfrac{V_0}{L}$ |
| spring and motion energy $\frac{1}{2}v^2 + \frac{1}{2}\omega_0^2x^2$ | capacitor and inductor energy $\frac{1}{2}LI^2 + \frac{q^2}{2C}$, divided by $L$ |
| power lost to friction $\gamma v^2$ | power lost in the resistor $I^2R$, divided by $L$ |
| power pushed in $f\cos(\Omega t)\,v$ | power from the source $V_0\cos(\Omega t)\,I$, divided by $L$ |
| quality factor $Q = \omega_0/\gamma$ | $Q = \dfrac{1}{R}\sqrt{\dfrac{L}{C}}$ |

The inductor plays the mass: it resists *changes* in current, the way a mass resists changes in velocity.  The capacitor plays the spring: the more charge you pile on, the harder it pushes back.  The resistor plays the friction, turning energy into heat.

So everything we found today carries over without changing a thing.
 - The circuit **resonates** at $\omega_0 = 1/\sqrt{LC}$, and the current there gets big, limited only by the resistance.
 - Right on resonance the charge lags a quarter cycle behind the source, which means the *current* is exactly in step with the source voltage.  That's exactly when the source delivers the most power.
 - The power the circuit absorbs is a **Lorentzian** in $\Omega$, with width $R/L$.

That's how an old-fashioned radio tunes in a station.  The antenna picks up *every* station at once, each a push at its own frequency.  Turning the dial changes $C$, which slides $\omega_0$ around.  The station whose frequency matches $\omega_0$ gets absorbed strongly, and the rest barely register.  A small $R$ (a high $Q$) makes a narrow peak, so neighboring stations don't bleed through.

# Summary

Today we pushed on a spring.
 - We split the push off into its own `Drive` object, so the spring doesn't care how it's pushed.
 - Push too slowly and the spring rides along with the push.  Push too fast and it barely moves, opposite the push.  Push near its natural frequency and the swing gets big: **resonance**.
 - We found the exact answers by **guess and check**: guess a form, plug it in, and pick the constants to make it work.  Adding a free swing fixes the starting point, because the equation is **linear**.
 - Without friction, pushing near resonance makes **beats**, and pushing exactly on resonance makes a swing that grows forever, with energy growing like $t^2$.
 - With friction, the start-up wiggles die away, leaving a steady swing.  The spring **lags** the push, and the lag sweeps from 0 to $\pi$, passing $\pi/2$ right on resonance.
 - Only the part of the motion a quarter cycle behind the push absorbs energy on average.  The average power absorbed is a **Lorentzian** whose width is the friction $\gamma$.
 - A velocity-dependent force **demotes** Verlet to first order.
 - The RLC circuit is the same equation with different letters.

Here's something to think about before next time.  You don't have to push a swing on *every* trip.  You could push it every *other* time it comes back, and it would still build up, as long as each push comes at the right moment.  But a push every other swing has a frequency of $\omega_0/2$, which is nowhere near resonance!  Our formulas say a smooth push at $\omega_0/2$ should never build up a big swing.  What's different about a real push?  It's not a smooth cosine.  It's a short shove.  To understand how a short shove can still resonate, we'll need to learn how to take a push apart into cosines.  That's **Fourier analysis**, and it's coming up.